In [21]:
import sys, os, importlib
package_path = os.path.abspath('../../')
if package_path not in sys.path:
    sys.path.append(package_path)
from package import functions as fn
from package import model as rm
from package import optimizer as opt
from package import plots
import numpy as np
import pandas as pd
import nbformat as nbf
from obspy.taup import TauPyModel
from obspy.geodetics import gps2dist_azimuth
from obspy.geodetics import kilometers2degrees
from obspy import read
from obspy import Stream
from matplotlib import pyplot as plt

from obspy.clients.fdsn import Client
from obspy import UTCDateTime
from obspy.signal.trigger import classic_sta_lta, trigger_onset
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

np.set_printoptions(precision=4, suppress=True)
for module in [fn, rm, opt, plots]:
    importlib.reload(module)

# set seed for reproducibility
np.random.seed(2026)

In [22]:
def find_extrema_indices(arr):
    """
    Returns indices of local maxima and minima.
    """
    extrema = []
    
    for i in range(1, len(arr) - 1):
        if arr[i] > arr[i-1] and arr[i] > arr[i+1]:
            extrema.append(i)
        elif arr[i] < arr[i-1] and arr[i] < arr[i+1]:
            extrema.append(i)
    
    return extrema


def average_half_period(arr):
    """
    Computes average spacing between consecutive extrema.
    This corresponds to average half-period.
    """
    extrema = find_extrema_indices(arr)
    
    if len(extrema) < 2:
        raise ValueError("Not enough extrema to compute half-period.")
    
    spacings = [extrema[i+1] - extrema[i] for i in range(len(extrema)-1)]
    
    return sum(spacings) / len(spacings)


def get_p_polarity(stream, p_inc, p_delay=300, window_size=60, stretch=None,
                   return_aic=False):
    # given one trace, get p wave polarity
    # use one loc per station for now
    
    baz = stream[0].stats.sac.baz
    p_init = stream[0].stats.starttime
    p_arrival = p_init + p_delay
    p_start = p_arrival - window_size
    p_end = p_arrival + window_size
    delta = stream[0].stats.delta
    
    data = np.zeros((len(stream), len(stream[0].data)))
    tr_ZL = stream.select(component='Z')[0].copy()
    tr_NQ = stream.select(component='N')[0].copy()
    tr_ET = stream.select(component='E')[0].copy()
    
    
    for tr in [tr_ZL, tr_NQ, tr_ET]: # IMPORTANT
        tr.detrend('linear')
        tr.detrend('demean')
        tr.filter('lowpass', freq=.1, corners=4,
                  zerophase=False)
    
    data[0, :] = tr_ZL.data
    data[1, :] = tr_NQ.data
    data[2, :] = tr_ET.data
    
    R_zne2zrt = fn.zne2zrt_matrix(baz, deg=True)
    R_zrt2lqt = fn.zrt2lqt_matrix(p_inc=p_inc, deg=True)
    data_lqt = R_zrt2lqt @ R_zne2zrt @ data
    
    pol_data = data_lqt[0, :]
    p_start_idx = int((p_start - p_init) / delta)
    p_end_idx = int((p_end - p_init) / delta)
    pol_window = pol_data[p_start_idx:p_end_idx+1]
    scale = 1/np.mean(np.abs(pol_window))
    aic_values = fn.aic(scale * pol_window)
    min_aic_idx = np.argmin(aic_values)
    
    quarter_period = average_half_period(pol_window)/2
    if stretch is None:
        stretch = int(quarter_period)
    
    zoom_pol_data = pol_window[min_aic_idx:min_aic_idx+stretch]
    diffs = scale * np.diff(zoom_pol_data)
    signs = np.sign(diffs)
    p_polarity = np.sign(np.sum(signs))
    
    if return_aic:
        return p_polarity, aic_values
    return p_polarity

In [23]:
data_path = '../../data/1994-01-17-mw67-southern-california'
files = [f for f in os.listdir(data_path) if f.endswith('.SAC')]
streams = [read(data_path + '/' + f) for f in files]
locs = {st[0].stats.location for st in streams}
print(locs)

{'', '00'}


In [24]:
# data preprocessing
# traces = [st[0] for st in streams if st[0].stats.location == '00']
traces = [st[0] for st in streams if st[0].stats.location == '00' or st[0].stats.location == '']
traces_dict = defaultdict(list)
for tr in traces: traces_dict[tr.stats.station].append(tr)

for station in list(traces_dict.keys()):
    if len(traces_dict[station]) != 3:
        print(f'{station} has {len(traces_dict[station])} trace(s).')
        del traces_dict[station]

station_count = 0
for name in traces_dict:
    station_count += 1
    if len(traces_dict[name]) != 3:
        print(f"Station {name} has {len(traces_dict[name])} trace(s).")
print(f'Total stations: {station_count}')

# NOTE: northridge
eq_lat = 34.136
eq_lon = -118.5826
hdepth = 15.9
 
velocity_model = TauPyModel(model='ak135')
vel_model_path = '../../data/AK135_lookup.csv'
lookup_table = pd.read_csv(vel_model_path)

alpha, beta = fn.extract_velocities(lookup_table, hdepth)
print(f'At depth {hdepth} km, P velocity = {alpha} km/s, S velocity = {beta} km/s')

TUC has 2 trace(s).
Total stations: 43
At depth 15.9 km, P velocity = 5.8 km/s, S velocity = 3.46 km/s


In [25]:
print(list(traces_dict.keys()))

['XAN', 'ARU', 'TATO', 'LVZ', 'COR', 'SPA', 'VNDA', 'NWAO', 'AFI', 'PMG', 'PMSA', 'LBTB', 'ERM', 'KEV', 'SJG', 'RPN', 'YSS', 'TAU', 'ABKT', 'RAR', 'TBT', 'YAK', 'GUMO', 'PET', 'CMB', 'ANMO', 'NNA', 'CHTO', 'ESK', 'CCM', 'AAK', 'PAB', 'HRV', 'KONO', 'ALE', 'ADK', 'COL', 'SNZO', 'CTAO', 'PFO', 'OBN', 'MAJO', 'KIV']


In [ ]:
col_string = 'event_id, station, network, location, channel, p_polarity, takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty'
columns = [col.strip() for col in col_string.split(',')]
df = pd.DataFrame(columns=columns)
event_id = 1
takeoff_uncertainty = 0.1 # placeholder
azimuth_uncertainty = 0.1 # placeholder
pre_filt = (0.01, 0.02, 8.0, 10.0)
aic_dict = {}
name_solid = None
count = 0

for name in traces_dict:
    count += 1
    if count % 10 == 0:
        print(f"Count = {count} for station {name}")
    stream = Stream(traces_dict[name])
    station = stream[0].stats.station
    network = stream[0].stats.network
    
    client = Client("IRIS")
    starttime = stream[0].stats.starttime
    endtime = stream[0].stats.endtime
    inv = client.get_stations(network=network, station=name,
                        starttime=starttime, endtime=endtime,
                        level='response')
    
    for trace in stream:
        trace.remove_response(inventory=inv, output="DISP", pre_filt=pre_filt,
                        plot=False, water_level=None)
    
    stream._rotate_to_zne(inventory=inv)
    
    # rotate the stream to ZNE
    location = stream[0].stats.location
    channel = stream[0].stats.channel
    azimuth = stream[0].stats.sac.az
    sta_lat = stream[0].stats.sac.stla
    sta_lon = stream[0].stats.sac.stlo
    event_locs = [eq_lat, eq_lon, sta_lat, sta_lon]
    dist, az_check, baz_check = gps2dist_azimuth(*event_locs)
    epdist = kilometers2degrees(dist/1000)
    p_arrivals = velocity_model.get_travel_times(source_depth_in_km=hdepth,
                        distance_in_degree=epdist, phase_list=['P'])
    
    if len(p_arrivals) == 0:
        print(f"Station {station} has no P arrival. Skipping.")
        print(f"Epidist = {epdist:.2f} degrees, depth = {hdepth} km")
        continue
    
    takeoff = p_arrivals[0].takeoff_angle
    
    p_arrival = p_arrivals[0]
    p_inc = p_arrival.incident_angle
    p_polarity, aic = get_p_polarity(stream=stream, p_inc=p_inc,
                                window_size=60, stretch=None, return_aic=True)
    aic_dict[station] = aic
    
    name_solid = name
    
    # now add row to pandas
    # p_polarity = -p_polarity # reverse to see NOTE
    df.loc[len(df)] = [event_id, station, network, location, channel, p_polarity,
                       takeoff, takeoff_uncertainty, azimuth, azimuth_uncertainty]
    
    # if count == 200:
    #     print(f'Stopping here')
    #     break


to_plot = aic_dict[name_solid]
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(np.arange(len(to_plot)), to_plot, 'k', label='AIC Function')
ax.set_xlabel('Time (s)')
ax.set_ylabel('AIC Value')
ax.set_title(f'Station {name_solid} - AIC Function for P Arrival')
ax.legend()
ax.grid()
plt.show()

KeyboardInterrupt: 

In [14]:
# save df as csv
# first check the directory we're in
print(os.getcwd())
# save to cwd, call it pol.csv
df.to_csv('pol.csv', index=False)

/home/vay3059/desktop/SURG-Mars/project/tests/actual


In [9]:
df

,event_id,station,network,location,channel,p_polarity,takeoff,takeoff_uncertainty,azimuth,azimuth_uncertainty
0,1,KIV,II,00,BHZ,-1.0,34.892811,0.1,85.032249,0.1
1,1,KMI,IC,00,BHZ,-1.0,18.483128,0.1,74.068207,0.1
2,1,HIA,IC,00,BHZ,1.0,19.475862,0.1,44.838287,0.1
3,1,MAKZ,IU,00,BHZ,-1.0,24.384203,0.1,63.565941,0.1
4,1,OBN,II,00,BHZ,1.0,39.704481,0.1,45.807205,0.1
...,...,...,...,...,...,...,...,...,...,...
86,1,GNI,IU,00,BHZ,-1.0,34.164428,0.1,94.076355,0.1
87,1,ANMO,IU,00,BHZ,1.0,14.766151,0.1,316.041931,0.1
88,1,SUR,II,00,BHZ,1.0,17.653414,0.1,176.055252,0.1
89,1,QIZ,IC,00,BHZ,-1.0,16.422077,0.1,74.027405,0.1


In [10]:
# find the takeoff and azimuth angles of selected stations
# MAKZ, ULN
takeoff_MAKZ = df[df['station'] == 'MAKZ']['takeoff'].values[0]
takeoff_ULN = df[df['station'] == 'ULN']['takeoff'].values[0]
azimuth_MAKZ = df[df['station'] == 'MAKZ']['azimuth'].values[0]
azimuth_ULN = df[df['station'] == 'ULN']['azimuth'].values[0]
print(f'MAKZ takeoff angle: {takeoff_MAKZ:.2f} degrees')
print(f'ULN takeoff angle: {takeoff_ULN:.2f} degrees')
print(f'MAKZ azimuth angle: {azimuth_MAKZ:.2f} degrees')
print(f'ULN azimuth angle: {azimuth_ULN:.2f} degrees')

MAKZ takeoff angle: 24.38 degrees
ULN takeoff angle: 20.88 degrees
MAKZ azimuth angle: 63.57 degrees
ULN azimuth angle: 51.93 degrees


In [ ]:
# select where takeoff is between 20 and 35 degrees
df_filtered = df[(df['takeoff'] >= 20) & (df['takeoff'] <= 90)]

# look at the takeoff angles
# takeoff_filtered = df_filtered['takeoff'].values
takeoff_filtered = df['takeoff'].values
# print(f'Filtered takeoff angles: {takeoff_filtered}')


df_filtered.to_csv('pol_filt.csv', index=False)
df_filtered

In [13]:
# get max takeoff angle's station
max_takeoff_idx = df_filtered['takeoff'].idxmax()
max_takeoff_station = df_filtered.loc[max_takeoff_idx, 'station']
max_takeoff_station

'KEV'